# 05 - Check whether the input-size result repeats

The 300-pixel model won on one validation split. Before trusting that result, this notebook compares it with the 224-pixel baseline across the remaining folds of a **5-fold stratified group split**. Exact copies stay in one fold.

Only IDRiD's official training portion is used here. The official test set is not opened in this notebook. Fold 1 results are reused from notebook 04; folds 2-5 are trained below. Each fold uses both input sizes with the same seed and settings, so the main change is image size.

This takes longer on a CPU. It creates separate files for every fold and can resume a fold by rerunning its cell. It does not replace the need for a new independent test dataset.

## 1. Load libraries and find the project

In [3]:
from pathlib import Path
import json, random, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score, confusion_matrix

print("PyTorch:", torch.__version__)
print("Device:", "CUDA" if torch.cuda.is_available() else "CPU")

PyTorch: 2.14.1+cpu
Device: CPU


## 2. Load only the eligible official training images

In [5]:
ROOT_OVERRIDE = None
if ROOT_OVERRIDE:
    ROOT = Path(ROOT_OVERRIDE).expanduser().resolve()
else:
    here = Path.cwd().resolve()
    choices = [here, *here.parents, Path.home() / "Desktop" / "retinal-dr-classification"]
    ROOT = next((p for p in choices if (p / "data" / "IDRiD").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Set ROOT_OVERRIDE to the project folder.")
RUN_DIR = ROOT / "reports" / "IDRiD_clean_run"
EXP_DIR = ROOT / "reports" / "IDRiD_experiments"
CV_DIR = ROOT / "reports" / "IDRiD_grouped_cv"
CV_DIR.mkdir(parents=True, exist_ok=True)

train_pool = pd.concat([
    pd.read_csv(RUN_DIR / "train_manifest.csv"),
    pd.read_csv(RUN_DIR / "validation_manifest.csv"),
], ignore_index=True).drop_duplicates("image_id").reset_index(drop=True)
print("Eligible official training images:", len(train_pool))
print("Labels: 0 = Normal; 1 = DR")
display(train_pool.label.value_counts().reindex([0, 1], fill_value=0).rename(index={0:"Normal",1:"DR"}))

Eligible official training images: 412
Labels: 0 = Normal; 1 = DR
label
Normal    133
DR        279


## 3. Make five grouped folds

Each image belongs to validation once. Exact copies that share a SHA-256 hash stay in the same fold. We check that the first fold matches the training/validation split already used in notebook 04.

In [7]:
SEED = 20261008
fold_maker = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
fold_rows = []
for fold_number, (train_idx, validation_idx) in enumerate(
    fold_maker.split(np.zeros(len(train_pool)), train_pool.label, groups=train_pool.sha256), start=1
):
    train_part = train_pool.iloc[train_idx].reset_index(drop=True)
    validation_part = train_pool.iloc[validation_idx].reset_index(drop=True)
    assert set(train_part.sha256).isdisjoint(set(validation_part.sha256))
    fold_rows.append((train_part, validation_part))
    print(f"Fold {fold_number}: {len(train_part)} train, {len(validation_part)} validation")

saved_fold1 = set(pd.read_csv(RUN_DIR / "validation_manifest.csv").image_id)
assert set(fold_rows[0][1].image_id) == saved_fold1
print("Fold 1 exactly matches the split used in notebook 04.")

Fold 1: 329 train, 83 validation
Fold 2: 330 train, 82 validation
Fold 3: 329 train, 83 validation
Fold 4: 330 train, 82 validation
Fold 5: 330 train, 82 validation
Fold 1 exactly matches the split used in notebook 04.


## 4. Keep the comparison settings fixed

Both models use ImageNet-pretrained EfficientNet-B0, class-weighted loss, Adam, the same batch size, the same maximum epochs, and horizontal flips. Only the input size changes.

In [9]:
MAX_EPOCHS = 8
BATCH_SIZE = 8
LEARNING_RATE = 0.0001
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]
INPUT_SIZES = {"224": 224, "300": 300}
print("Fixed settings:", MAX_EPOCHS, "epochs; batch", BATCH_SIZE, "; learning rate", LEARNING_RATE)

Fixed settings: 8 epochs; batch 8 ; learning rate 0.0001


## 5. Prepare a fold of images

In [11]:
class FundusDataset(Dataset):
    def __init__(self, table, image_size, training=False):
        self.table = table.reset_index(drop=True)
        steps = [transforms.Resize(image_size + 32), transforms.CenterCrop(image_size)]
        if training:
            steps.append(transforms.RandomHorizontalFlip(p=0.5))
        steps += [transforms.ToTensor(), transforms.Normalize(MEAN, STD)]
        self.transform = transforms.Compose(steps)

    def __len__(self):
        return len(self.table)

    def __getitem__(self, index):
        row = self.table.iloc[index]
        with Image.open(ROOT / row.relative_path) as image:
            image = image.convert("RGB")
        return self.transform(image), int(row.label), row.image_id

def make_fold_loaders(train_part, validation_part, image_size):
    train_loader = DataLoader(FundusDataset(train_part, image_size, True),
                              batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    validation_loader = DataLoader(FundusDataset(validation_part, image_size, False),
                                   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    return train_loader, validation_loader
print("Image loaders are ready.")

Image loaders are ready.


## 6. Define scoring and the training function

In [13]:
def score_predictions(labels, probabilities):
    predicted = (probabilities >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, predicted, labels=[0, 1]).ravel()
    return {"accuracy": accuracy_score(labels, predicted),
            "sensitivity": recall_score(labels, predicted, zero_division=0),
            "specificity": tn / (tn + fp), "roc_auc": roc_auc_score(labels, probabilities),
            "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp)}

def get_probabilities(model, loader):
    model.eval(); labels=[]; probabilities=[]; image_ids=[]
    with torch.inference_mode():
        for images, batch_labels, ids in loader:
            logits=model(images.to(DEVICE))
            probabilities.extend(torch.softmax(logits,dim=1)[:,1].cpu().numpy())
            labels.extend(batch_labels.numpy()); image_ids.extend(ids)
    return np.asarray(labels), np.asarray(probabilities), image_ids

def train_fold_model(fold_number, train_part, validation_part, image_size):
    seed = SEED + fold_number - 1
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    train_loader, validation_loader = make_fold_loaders(train_part, validation_part, image_size)
    model = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, 2)
    model = model.to(DEVICE)

    counts = train_part.label.value_counts().reindex([0,1],fill_value=0)
    weights = len(train_part) / (2 * counts.to_numpy(dtype=np.float32))
    loss_function = nn.CrossEntropyLoss(weight=torch.tensor(weights,dtype=torch.float32,device=DEVICE))
    optimizer = torch.optim.Adam(model.parameters(),lr=LEARNING_RATE)
    best_loss=float('inf'); best_epoch=0
    name=f"fold{fold_number}_{image_size}px_weighted"
    model_path=CV_DIR/f"{name}.pt"; history=[]

    for epoch in range(1,MAX_EPOCHS+1):
        model.train(); loss_total=0.0; seen=0
        for images,labels,_ in train_loader:
            images=images.to(DEVICE); labels=labels.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            loss=loss_function(model(images),labels); loss.backward(); optimizer.step()
            loss_total += loss.item()*len(labels); seen += len(labels)
        model.eval(); val_loss_total=0.0; val_seen=0
        with torch.inference_mode():
            for images,labels,_ in validation_loader:
                images=images.to(DEVICE); labels=labels.to(DEVICE)
                val_loss_total += nn.functional.cross_entropy(model(images),labels,reduction='sum').item()
                val_seen += len(labels)
        val_loss=val_loss_total/val_seen
        history.append({"epoch":epoch,"training_loss":loss_total/seen,"validation_loss":val_loss})
        print(f"Fold {fold_number}, {image_size}px, epoch {epoch}/{MAX_EPOCHS}: validation loss {val_loss:.3f}",flush=True)
        if val_loss < best_loss:
            best_loss=val_loss; best_epoch=epoch
            torch.save(model.state_dict(),model_path)

    model.load_state_dict(torch.load(model_path,map_location=DEVICE,weights_only=True))
    labels,probabilities,ids=get_probabilities(model,validation_loader)
    metrics=score_predictions(labels,probabilities)
    result={"fold":fold_number,"input_size":image_size,"best_epoch":best_epoch,
            "best_validation_loss":best_loss,**metrics}
    pd.DataFrame(history).to_csv(CV_DIR/f"{name}_history.csv",index=False)
    pd.DataFrame({"image_id":ids,"true_label":labels,"dr_probability":probabilities,
                  "predicted_label":(probabilities>=0.5).astype(int)}).to_csv(
                  CV_DIR/f"{name}_predictions.csv",index=False)
    (CV_DIR/f"{name}_config.json").write_text(json.dumps({
        "seed":seed,"epochs":MAX_EPOCHS,"batch_size":BATCH_SIZE,
        "learning_rate":LEARNING_RATE,"class_weighted":True,
        "input_size":image_size,"checkpoint_rule":"lowest validation loss"},indent=2),encoding='utf-8')
    return result
print("Fold training function is ready.")

Fold training function is ready.


## 7. Add fold 1 results from the first controlled experiment

We reuse fold 1 rather than retraining it. This file was created by notebook 04 using the same 329/83 split and weighted 224/300 input setups.

In [15]:
first_results = pd.read_csv(EXP_DIR / "validation_experiment_comparison.csv")
fold1 = first_results[first_results.name.isin(
    ["baseline_224_weighted", "larger_input_300_weighted"]
)].copy()
fold1["fold"] = 1
fold1["input_size"] = fold1.input_size.astype(int)
fold1 = fold1[["fold","input_size","best_epoch","best_validation_loss",
               "accuracy","sensitivity","specificity","roc_auc","TN","FP","FN","TP"]]
print("Fold 1 reused from notebook 04:")
display(fold1)

Fold 1 reused from notebook 04:
   fold  input_size  best_epoch  best_validation_loss  accuracy  sensitivity  specificity   roc_auc  TN  FP  FN  TP
0     1         224           6              0.169400  0.915663     0.892857     0.962963  0.984127  26   1   6  50
2     1         300           7              0.169535  0.927711     0.928571     0.925926  0.982143  25   2   4  52


## 8. Train fold 2 with both input sizes

The fold is split by exact-image groups. Both models use the same fold and seed.

In [17]:
train_part, validation_part = fold_rows[1]
fold2_results = []
for image_size in [224, 300]:
    fold2_results.append(train_fold_model(2, train_part, validation_part, image_size))
fold2_table = pd.DataFrame(fold2_results)
fold2_table.to_csv(CV_DIR / "fold2_results.csv", index=False)
display(fold2_table)

Fold 2, 224px, epoch 1/8: validation loss 0.452
Fold 2, 224px, epoch 2/8: validation loss 0.386
Fold 2, 224px, epoch 3/8: validation loss 0.334
Fold 2, 224px, epoch 4/8: validation loss 0.309
Fold 2, 224px, epoch 5/8: validation loss 0.337
Fold 2, 224px, epoch 6/8: validation loss 0.334
Fold 2, 224px, epoch 7/8: validation loss 0.364
Fold 2, 224px, epoch 8/8: validation loss 0.329
Fold 2, 300px, epoch 1/8: validation loss 0.420
Fold 2, 300px, epoch 2/8: validation loss 0.341
Fold 2, 300px, epoch 3/8: validation loss 0.277
Fold 2, 300px, epoch 4/8: validation loss 0.267
Fold 2, 300px, epoch 5/8: validation loss 0.286
Fold 2, 300px, epoch 6/8: validation loss 0.232
Fold 2, 300px, epoch 7/8: validation loss 0.261
Fold 2, 300px, epoch 8/8: validation loss 0.303
   fold  input_size  best_epoch  best_validation_loss  accuracy  sensitivity  specificity   roc_auc  TN  FP  FN  TP
0     2         224           4              0.308613  0.865854     0.857143     0.884615  0.949863  23   3   8  48


## 9. Train fold 3 with both input sizes

The fold is split by exact-image groups. Both models use the same fold and seed.

In [19]:
train_part, validation_part = fold_rows[2]
fold3_results = []
for image_size in [224, 300]:
    fold3_results.append(train_fold_model(3, train_part, validation_part, image_size))
fold3_table = pd.DataFrame(fold3_results)
fold3_table.to_csv(CV_DIR / "fold3_results.csv", index=False)
display(fold3_table)

Fold 3, 224px, epoch 1/8: validation loss 0.420
Fold 3, 224px, epoch 2/8: validation loss 0.367
Fold 3, 224px, epoch 3/8: validation loss 0.275
Fold 3, 224px, epoch 4/8: validation loss 0.296
Fold 3, 224px, epoch 5/8: validation loss 0.261
Fold 3, 224px, epoch 6/8: validation loss 0.292
Fold 3, 224px, epoch 7/8: validation loss 0.265
Fold 3, 224px, epoch 8/8: validation loss 0.298
Fold 3, 300px, epoch 1/8: validation loss 0.384
Fold 3, 300px, epoch 2/8: validation loss 0.343
Fold 3, 300px, epoch 3/8: validation loss 0.231
Fold 3, 300px, epoch 4/8: validation loss 0.267
Fold 3, 300px, epoch 5/8: validation loss 0.259
Fold 3, 300px, epoch 6/8: validation loss 0.339
Fold 3, 300px, epoch 7/8: validation loss 0.271
Fold 3, 300px, epoch 8/8: validation loss 0.306
   fold  input_size  best_epoch  best_validation_loss  accuracy  sensitivity  specificity   roc_auc  TN  FP  FN  TP
0     3         224           5              0.261035  0.915663     0.928571     0.888889  0.956349  24   3   4  52


## 10. Train fold 4 with both input sizes

The fold is split by exact-image groups. Both models use the same fold and seed.

In [21]:
train_part, validation_part = fold_rows[3]
fold4_results = []
for image_size in [224, 300]:
    fold4_results.append(train_fold_model(4, train_part, validation_part, image_size))
fold4_table = pd.DataFrame(fold4_results)
fold4_table.to_csv(CV_DIR / "fold4_results.csv", index=False)
display(fold4_table)

Fold 4, 224px, epoch 1/8: validation loss 0.436
Fold 4, 224px, epoch 2/8: validation loss 0.401
Fold 4, 224px, epoch 3/8: validation loss 0.330
Fold 4, 224px, epoch 4/8: validation loss 0.436
Fold 4, 224px, epoch 5/8: validation loss 0.306
Fold 4, 224px, epoch 6/8: validation loss 0.286
Fold 4, 224px, epoch 7/8: validation loss 0.362
Fold 4, 224px, epoch 8/8: validation loss 0.518
Fold 4, 300px, epoch 1/8: validation loss 0.388
Fold 4, 300px, epoch 2/8: validation loss 0.388
Fold 4, 300px, epoch 3/8: validation loss 0.311
Fold 4, 300px, epoch 4/8: validation loss 0.398
Fold 4, 300px, epoch 5/8: validation loss 0.331
Fold 4, 300px, epoch 6/8: validation loss 0.301
Fold 4, 300px, epoch 7/8: validation loss 0.286
Fold 4, 300px, epoch 8/8: validation loss 0.386
   fold  input_size  best_epoch  best_validation_loss  accuracy  sensitivity  specificity   roc_auc  TN  FP  FN  TP
0     4         224           6              0.286456  0.853659     0.818182     0.925926  0.962963  25   2  10  45


## 11. Train fold 5 with both input sizes

The fold is split by exact-image groups. Both models use the same fold and seed.

In [ ]:
train_part, validation_part = fold_rows[4]
fold5_results = []
for image_size in [224, 300]:
    fold5_results.append(train_fold_model(5, train_part, validation_part, image_size))
fold5_table = pd.DataFrame(fold5_results)
fold5_table.to_csv(CV_DIR / "fold5_results.csv", index=False)
display(fold5_table)

## 12. Compare the five folds

The average and spread across folds are more useful than one validation split. A large spread means performance depends on which images land in validation. The official test set is not used.

In [ ]:
all_results = [fold1]
for fold_number in range(2,6):
    fold_file = CV_DIR / f"fold{fold_number}_results.csv"
    if not fold_file.exists():
        raise FileNotFoundError(f"Finish fold {fold_number} before this comparison.")
    all_results.append(pd.read_csv(fold_file))
cv_results = pd.concat(all_results,ignore_index=True).sort_values(["input_size","fold"])
summary = cv_results.groupby("input_size").agg(
    accuracy_mean=("accuracy","mean"), accuracy_sd=("accuracy","std"),
    sensitivity_mean=("sensitivity","mean"), sensitivity_sd=("sensitivity","std"),
    specificity_mean=("specificity","mean"), specificity_sd=("specificity","std"),
    auc_mean=("roc_auc","mean"), auc_sd=("roc_auc","std"),
).reset_index()
display(cv_results[["fold","input_size","best_epoch","accuracy","sensitivity",
                    "specificity","roc_auc","TN","FP","FN","TP"]])
display(summary)
cv_results.to_csv(CV_DIR/"all_fold_results.csv",index=False)
summary.to_csv(CV_DIR/"cross_validation_summary.csv",index=False)
print("Do the mean AUC and sensitivity favor the same size? Compare the fold-by-fold values before deciding.")

## What this check can tell us

If 300 pixels wins on most folds and its average results are better, the input-size choice is more convincing. If results switch back and forth, the dataset is too small for a confident choice. This still estimates performance using IDRiD only; it is not an independent external test. A qualified reviewer should check the saved validation-error contact sheet for labeling or image-quality concerns. No test-set results are used to make a model choice here.